# Logistic Regression Modeling (FAST + STABLE)

MNIST Classification using Logistic Regression, L1 and L2 Regularization with Cross-Validation.

## 1. Import Libraries

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Helper Functions

In [ ]:
def log(msg):
    print(f"\n[INFO] {msg}")

def log_time(start, label):
    print(f"[DONE] {label} in {time.time() - start:.2f} seconds")

def evaluate(name, y_true, y_pred):
    print(f"\n=========================")
    print(name)
    print("=========================")

    acc = accuracy_score(y_true, y_pred)
    print(f"Accuracy: {acc:.4f}")

    report = classification_report(y_true, y_pred, output_dict=True)
    df_report = pd.DataFrame(report).transpose()
    print("\nClassification Report (Table):")
    display(df_report)

    cm = confusion_matrix(y_true, y_pred)

    plt.figure()
    plt.imshow(cm)
    plt.title(f"{name} - Confusion Matrix")
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.colorbar()

    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    plt.tight_layout()
    plt.show()

## 3. Load Dataset

In [ ]:
start = time.time()
log("Loading MNIST dataset...")

mnist = fetch_openml('mnist_784', version=1)

X = mnist.data
y = mnist.target.astype(int)

log_time(start, "Dataset Loaded")

## 4. Reduce Dataset Size

In [ ]:
log("Reducing dataset size for faster computation...")
X = X[:10000]
y = y[:10000]

## 5. Preprocessing

In [ ]:
start = time.time()
log("Preprocessing data...")

X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

log_time(start, "Preprocessing")

## 6. Simple Logistic Regression

In [ ]:
start = time.time()
log("Training Simple Logistic Regression...")

simple_model = LogisticRegression(
    solver='lbfgs',
    max_iter=300,
    tol=1e-2,
    verbose=1
)

simple_model.fit(X_train, y_train)

log_time(start, "Simple Model")

y_pred_simple = simple_model.predict(X_test)

## 7. Logistic Regression (L1 + CV)

In [ ]:
start = time.time()
log("Training Logistic Regression (L1 + CV)...")

l1_model = LogisticRegressionCV(
    penalty='l1',
    solver='saga',
    cv=3,
    Cs=5,
    max_iter=300,
    tol=1e-2,
    n_jobs=-1,
    verbose=1
)

l1_model.fit(X_train, y_train)

log_time(start, "L1 + CV Model")

y_pred_l1 = l1_model.predict(X_test)

## 8. Logistic Regression (L2 + CV)

In [ ]:
start = time.time()
log("Training Logistic Regression (L2 + CV)...")

l2_model = LogisticRegressionCV(
    penalty='l2',
    solver='saga',
    cv=3,
    Cs=5,
    max_iter=300,
    tol=1e-2,
    n_jobs=-1,
    verbose=1
)

l2_model.fit(X_train, y_train)

log_time(start, "L2 + CV Model")

y_pred_l2 = l2_model.predict(X_test)

## 9. Evaluation

In [ ]:
log("Evaluating models...")

evaluate("Simple Logistic Regression", y_test, y_pred_simple)
evaluate("Logistic Regression (L1 + CV)", y_test, y_pred_l1)
evaluate("Logistic Regression (L2 + CV)", y_test, y_pred_l2)